In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent))
import config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv(config.DB_LOCATION)

C:\Users\bnpar\AppData\Local\Temp\ipykernel_30884\998648215.py:12: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(config.DB_LOCATION)


This notebook performs data cleaning to ensure the dataset is consistent. This includes ensuring consistent data types and resolving inconsistencies such as incorrect weight class entries.

In [2]:
#reduced dataframe
red_df = df[['Name','Date','Sex', 'Age', 'MeetName', 'Federation','Division', 'BodyweightKg', 'WeightClassKg',
         'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 
         'Equipment', 'Event', 
         'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 
         'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 
         'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg',
        'TotalKg', 'Goodlift', 'Sanctioned', 'Country']]

In [ ]:
def get_mixed_cols(input_df):
    """Return columns containing more than one data type."""
    mixed_cols = []
    for col in input_df.columns:
        if input_df[col].map(type).nunique() > 1:
            mixed_cols.append(col)
    return mixed_cols

mixed_cols = get_mixed_cols(red_df)
print(mixed_cols)

In [ ]:
#checking if all the floats are because of NaN
for col in mixed_cols:
    mask = df[col].map(type) == float
    float_col = df.loc[mask, col].dropna()
    if len(float_col)==0:
        print(f'all floats in {col} are because of NaN')
    else:
        print(f'{col} has mixed datatypes')
        

Columns with mixed datatypes were found to contain floats and strings. In columns with both strings and floats, float datatypes were found to all be NaN entries

- The Division column contained a large number of unique values (1135), likely due to inconsistent data entry, making it unsuitable as a categorical feature. The column was therefore dropped.
- Rows with missing WeightClassKg were dropped. (1.17% of orginal dataset)
- Chosen to only include Sanctioned meets in dataset (specified in 'Sanctioned' column)
- Only raw powerlifting entries were retained, as equipped powerlifting is often considered a distinct discipline and may not be directly comparable (as indicated by the Equipment column).
- Filtered for full power events (squat, bench, and deadlift as indicated by the 'Event' column), as single-lift competitions (e.g. bench-only) represent different participation patterns.

In [ ]:
red_df.loc[:, 'Name'] = red_df['Name'].str.strip().str.lower()
red_df.loc[:, 'MeetName'] = red_df['MeetName'].str.strip().str.lower()
red_df.loc[:, 'Federation'] = red_df['Federation'].str.strip().str.lower()
red_df.loc[:, 'Equipment'] = red_df['Equipment'].str.strip().str.lower()
red_df.loc['WeightClassKg'] = red_df['WeightClassKg'].str.strip().str.lower()


red_df = red_df.loc[red_df['Equipment'] == 'raw']
red_df['Date'] = pd.to_datetime(red_df['Date'] , format = '%Y-%m-%d')
red_df['Year'] = red_df['Date'].dt.year
red_df = red_df.dropna(axis = 'rows', subset = ['WeightClassKg'])
red_df = red_df.loc[red_df['Sanctioned'] != 'No'].copy()

# Given powerlifters compete only a few times per year, entries with the same name, sex, event, date, total, and meet are likely to be duplicate records.
# A subset of columns is used rather than all columns to capture duplicates despite minor inconsistencies in other fields.
red_df = red_df.drop_duplicates(['Name' ,'Sex', 'Event', 'Date', 'TotalKg', 'MeetName'])

full_power = red_df[red_df['Event'] == 'SBD'].copy()

Entries exceeding current official world records were flagged for inspection and can be seen in exceeds_wr. Note that exceeding an official world record does not necessarily indicate an error, as records can only be set at international competitions, and higher totals may occur at national or local events without being recognised as official records.




In [ ]:
m_off_wr_totals = {'53': 559, '59': 669.5, '66': 770, '74': 843, '83': 890, '93': 918, '105': 975.5, '120': 978.5, '120+': 1152.5}
f_off_wr_totals = {'43': 349.5, '47': 435, '52': 482.5, '57': 520, '63': 565, '69': 628, '76': 620, '84': 647, '84+': 737.5}

m_cl = list(m_off_wr_totals.keys())
f_cl = list(f_off_wr_totals.keys())
wr_df = pd.concat([
    pd.DataFrame({'WeightClassKg': m_cl, 'Sex': ['M']* len(m_cl), 'official_wr_total': list(m_off_wr_totals.values())}),
    pd.DataFrame({'WeightClassKg': f_cl,'Sex': ['F']*len(f_cl),'official_wr_total': list(f_off_wr_totals.values())})
], ignore_index=True)


full_power_wr = full_power.merge(wr_df, on=['Sex', 'WeightClassKg'], how='left')
exceeds_wr = full_power_wr.loc[full_power_wr['TotalKg'] > full_power_wr['official_wr_total']]


Entries exceeding current world records were reviewed against available meet results and appeared credible.

### Dealing with changes in IPF weight classes

Next, we identify weight classes in the dataset that do not correspond to those currently used by the IPF and its affiliates.

#### F weight classes

In [ ]:
current_f_class_mask = (full_power['WeightClassKg'].isin(f_off_wr_totals.keys())) & (full_power['Sex'] == 'F')
current_m_class_mask = (full_power['WeightClassKg']).isin(m_off_wr_totals.keys()) & (full_power['Sex'] == 'M')

invalid_class_entries = full_power[(~current_f_class_mask) & (~current_m_class_mask) ].copy()
f_invalid_class_entries = full_power[~current_f_class_mask].copy()
m_invalid_class_entries = full_power[~current_m_class_mask].copy()

current_f_classes = full_power[current_f_class_mask].copy()
current_m_classes = full_power[current_m_class_mask].copy()

f_annual_per_current_class = current_f_classes.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'AnnualClassParticipation')
total_annual = full_power[full_power['Sex']== 'F'].groupby('Year').size().reset_index(name = 'AnnualParticipation') #total annual participation across weight classes (including historic and mistaken weight classes) 

f_annual_per_current_class = f_annual_per_current_class.merge(total_annual, on = 'Year', how = 'left', validate = 'm:1') 

#now we want annual participation excl historical/invlaid classes
f_current_class_share = f_annual_per_current_class.groupby('Year')['AnnualClassParticipation'].sum().reset_index(name = 'CurrentClassParticipation')

f_annual_per_current_class = f_annual_per_current_class.merge(f_current_class_share, how = 'left', on = 'Year')
f_annual_per_current_class['CurrentClassShare'] = (f_annual_per_current_class['CurrentClassParticipation']/f_annual_per_current_class['AnnualParticipation'])*100

df = f_annual_per_current_class.copy() 
df_sorted = df.sort_values('Year')
plt.plot(df_sorted['Year'],df_sorted['CurrentClassShare'], marker = 'o')
plt.title("Percentage of female entries made up by current IPF female weight classes by year")
plt.ylabel("Share made up by current IPF weight classes (%)")
plt.xlabel("Year")
plt.show()

Graph indicates weight classes changed between 2020 and 2021. This was analysed further below to determine the weight classes present in 2015 to 2020.

In [ ]:
old_struct_years = [2015, 2016, 2017, 2018, 2019, 2020]
new_struct_years = [2021, 2022, 2023,2024, 2025]
all_years = old_struct_years + new_struct_years
f = full_power.loc[(full_power['Sex'] == 'F') & (full_power['Year']).isin(all_years)]


f_class_participation = f.groupby(['WeightClassKg', 'Sex', 'Year']).size().reset_index(name = 'AnnualClassParticipation')
f_total_participation = f_class_participation.groupby('Year')['AnnualClassParticipation'].sum().reset_index(name = 'AnnualParticipation')

f_class_participation = f_class_participation.merge(f_total_participation, on = 'Year', how = 'left', validate = 'm:1')
f_class_participation['ClassShare'] = (f_class_participation['AnnualClassParticipation']/ f_class_participation['AnnualParticipation'])*100



In [ ]:
#looking at class participation in weight classes that are NOT the current classes 
old = f_class_participation.loc[f_class_participation['Year'].isin(old_struct_years)]
old_copy = old.loc[~old["WeightClassKg"].isin(f_off_wr_totals.keys())].copy() 


# keep only the year with the maximum class share for each weight class
old_copy = old_copy.loc[
    old_copy.groupby("WeightClassKg")["ClassShare"].idxmax()
].copy()


rng = np.random.default_rng(42)
old_copy["y_jitter"] = rng.uniform(-0.4, 0.4, size=len(old_copy))

plt.figure()
sns.scatterplot(
    data=old_copy,
    x="ClassShare",
    y="y_jitter",
    hue="WeightClassKg",
    alpha=0.8,
    s=60
)

plt.xlabel("ClassShare (%)")
plt.yticks([])
plt.ylabel("")
plt.title("Maximum class share by weight class 2015-2020  for classes which are not current IPF classes\n"
         "The points correspond to the year in which the class had its highest representation",
 )
plt.legend(title="WeightClassKg", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

Most weight classes that do not correspond to the current IPF structure have negligible representation. One exception is a class with substantially higher participation, which corresponds to the 72kg weight class. (This was verified to account for ambiguity in the colour mapping.)  
The 72kg class was present between 2015 and 2020. Next, weight classes present in 2021–2025 but absent in 2015–2020 were identified.

In [ ]:
#looking at changes in weight class prescence in dataset for candidate weight classes (i.e. all current weight classes AND 72kg class)

current_w_cl = list(f_off_wr_totals.keys())
candidate_old_wcl = current_w_cl + ['72']

relevant_years = (f_class_participation.loc[f_class_participation["Year"].isin([2020, 2021]) & (f_class_participation['WeightClassKg'].isin(candidate_old_wcl)),["WeightClassKg", "Year", "ClassShare"]])
change_in_class = relevant_years.pivot(index = 'WeightClassKg', columns = 'Year', values = 'ClassShare')
change_in_class = change_in_class.rename(columns = {2020: "ClassShare_2020", 2021: "ClassShare_2021"})
change_in_class["Absolute_Change"] = change_in_class['ClassShare_2021'] - change_in_class['ClassShare_2020']

change_in_class = change_in_class.sort_values("Absolute_Change", ascending=False)
change_in_class

69kg and 76kg classes show a sharp increase from negligible to significant representation, suggesting these weight classes were added between 2020 and 2021. (The 72kg class shows the opposite pattern, consistent with its removal as previously determined.)

old_classes = ['43', '47', '52', '57', '63', '72', '84', '84+']  
new_classes = ['43', '47', '52', '57', '63', '69','76', '84', '84+']

Removed entries where the weight class does not match those used at the time.


In [ ]:
old_classes = ['43', '47', '52', '57', '63', '72', '84', '84+']
new_classes = ['43', '47', '52', '57', '63', '69','76', '84', '84+']

f_cleaned_classes = f.loc[
    (f['Year'].isin(old_struct_years)& f['WeightClassKg'].isin(old_classes))|
    (f['Year'].isin(new_struct_years)& f['WeightClassKg'].isin(new_classes))
    ]

In [ ]:
f'{(len(f)/len(full_power[full_power['Sex']=='F']))*100:.2f}% of F IPF entries are in the last 10 years'

#### M weight classes

Percentage of male entries made up by current IPF female weight classes was plotted by year to examine changes in weight classes.

In [ ]:
current_m_class_mask = (full_power['WeightClassKg']).isin(m_off_wr_totals.keys()) & (full_power['Sex'] == 'M')

invalid_class_entries = full_power[(~current_f_class_mask) & (~current_m_class_mask) ].copy()
m_invalid_class_entries = full_power[~current_m_class_mask].copy()

current_m_classes = full_power[current_m_class_mask].copy()

m_annual_per_current_class = current_m_classes.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'AnnualClassParticipation')
total_annual = full_power[full_power['Sex']== 'M'].groupby('Year').size().reset_index(name = 'AnnualParticipation') #total annual participation across weight classes (including historic and mistaken weight classes) 

m_annual_per_current_class = m_annual_per_current_class.merge(total_annual, on = 'Year', how = 'left', validate = 'm:1') 

#now we want annual participation excl incurrent classes
m_current_class_share = m_annual_per_current_class.groupby('Year')['AnnualClassParticipation'].sum().reset_index(name = 'CurrentClassParticipation')

m_annual_per_current_class = m_annual_per_current_class.merge(m_current_class_share, how = 'left', on = 'Year')
m_annual_per_current_class['CurrentClassShare'] = (m_annual_per_current_class['CurrentClassParticipation']/m_annual_per_current_class['AnnualParticipation'])*100

df = m_annual_per_current_class.copy() 
df_sorted = df.sort_values('Year')
plt.plot(df_sorted['Year'],df_sorted['CurrentClassShare'], marker = 'o')
plt.title("Percenatge of male entries made up by current IPF male weight classes by year")
plt.ylabel("Current Class Share (%)")
plt.show()

Shows weight class restructuring in 2015. Can use data from 2015 to 2025 matching the time span for the F dataset.

In [ ]:
m = full_power.loc[(full_power['Sex'] == 'M') & (full_power['Year']).isin(all_years)]

old_struct_years = [2015, 2016, 2017, 2018, 2019, 2020]
new_struct_years = [2021, 2022, 2023,2024, 2025]
all_years = old_struct_years + new_struct_years
all_mens_classes = ['53', '59', '66', '74', '83', '93', '105', '120', '120+']

m_cleaned_classes = m.loc[
    (m['Year'].isin(all_years)& m['WeightClassKg'].isin(all_mens_classes))
    ]

The current IPF rules define men’s and women’s weight categories, with no non-binary division listed. Entries recorded as Mx in the Sex field were therefore retained only where their WeightClassKg matched a weight class used elsewhere in the men’s or women’s divisions in the dataset.

In [ ]:
mx = full_power.loc[(full_power['Sex'] == 'Mx') & (full_power['Year']).isin(all_years)]
all_classes = list(f_cleaned_classes['WeightClassKg'].unique()) + list(m_cleaned_classes['WeightClassKg'].unique())
mx_cleaned_classes = mx[mx['WeightClassKg'].isin(all_classes)]

In [ ]:
cleaned = pd.concat([f_cleaned_classes, m_cleaned_classes, mx_cleaned_classes], ignore_index = True)
cleaned_path = config.PROJECT_ROOT/ "data"/"processed" / "cleaned_comp_hist.csv"
cleaned.to_csv(cleaned_path, index = False)

In [ ]:
full_power_path = config.PROJECT_ROOT/"data"/"processed"/"all_years.csv"
full_power.to_csv(full_power_path, index = False)